In [3]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Lab2-Transactions")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — gotowy")

ModuleNotFoundError: No module named 'pyspark'

In [5]:
df = spark.read.json("data/transactions_10k.jsonl")

print(f"Liczba rekordów: {df.count()}")
df.printSchema()

Liczba rekordów: 10000
root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [6]:
df.show(10, truncate=False)

+------+-----------+--------+-------------------+-------+-------+
|amount|category   |store   |timestamp          |tx_id  |user_id|
+------+-----------+--------+-------------------+-------+-------+
|312.32|elektronika|Warszawa|2026-04-12 08:25:07|TX00001|u48    |
|79.57 |książki    |Warszawa|2026-04-12 08:05:43|TX00002|u15    |
|126.17|odzież     |Warszawa|2026-04-12 09:15:30|TX00003|u18    |
|34.08 |odzież     |Warszawa|2026-04-12 10:05:39|TX00004|u10    |
|428.88|żywność    |Kraków  |2026-04-12 09:04:36|TX00005|u17    |
|345.21|książki    |Warszawa|2026-04-12 09:36:31|TX00006|u25    |
|376.42|żywność    |Warszawa|2026-04-12 10:06:49|TX00007|u15    |
|85.36 |elektronika|Gdańsk  |2026-04-12 09:08:25|TX00008|u24    |
|66.26 |żywność    |Kraków  |2026-04-12 10:06:19|TX00009|u05    |
|660.41|odzież     |Kraków  |2026-04-12 08:29:24|TX00010|u41    |
+------+-----------+--------+-------------------+-------+-------+
only showing top 10 rows



In [7]:
from pyspark.sql.functions import to_timestamp, col

df = df.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))

df.printSchema()  # timestamp powinien być teraz 'timestamp (nullable = true)'

root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [18]:
from pyspark.sql.functions import count, sum as _sum, avg, round as _round

store_summary = (
    df.groupBy("store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(avg("amount"), 2).alias("srednia_PLN"),
    )
    .orderBy("store")
)
store_summary.show()

+--------+---------+----------+-----------+
|   store|liczba_tx|  suma_PLN|srednia_PLN|
+--------+---------+----------+-----------+
|  Gdańsk|     2498|1021266.35|     408.83|
|  Kraków|     2522|1025896.95|     406.78|
|Warszawa|     2424| 961642.24|     396.72|
| Wrocław|     2556|1002739.21|     392.31|
+--------+---------+----------+-----------+



In [8]:
from pyspark.sql.functions import min as _min, max as _max, sum as _sum, round as _round

category_stats = (
    df.groupBy("category")
    .agg(
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(_min("amount"), 2).alias("min_PLN"),
        _round(_max("amount"), 2).alias("max_PLN"),
    )
    .orderBy("category")
)

category_stats.show()

+-----------+----------+-------+-------+
|   category|  suma_PLN|min_PLN|max_PLN|
+-----------+----------+-------+-------+
|elektronika|1520770.69|    9.0| 9999.0|
|    książki| 851382.08|    5.0|9107.25|
|     odzież| 849877.55|    5.0|9696.63|
|    żywność| 789514.43|    5.0|6916.92|
+-----------+----------+-------+-------+



In [9]:
from pyspark.sql.functions import window

hourly = (
    df.groupBy(window("timestamp", "1 hour"))    # okno 1-godzinne
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy("window")
)
hourly.show(truncate=False)

+------------------------------------------+---------+----------+
|window                                    |liczba_tx|suma_PLN  |
+------------------------------------------+---------+----------+
|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|3150     |1241911.3 |
|{2026-04-12 09:00:00, 2026-04-12 10:00:00}|4661     |1896230.21|
|{2026-04-12 10:00:00, 2026-04-12 11:00:00}|2189     |873403.24 |
+------------------------------------------+---------+----------+



In [11]:
from pyspark.sql.functions import col

(
    hourly
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .show(truncate=False)
)

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150     |1241911.3 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661     |1896230.21|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189     |873403.24 |
+-------------------+-------------------+---------+----------+



In [12]:
from pyspark.sql.functions import window, count, sum as _sum, round as _round

per_store_30m = (
    df.groupBy(window("timestamp", "30 minutes"), "store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy("window", "store")
)

per_store_30m.show(truncate=False)

+------------------------------------------+--------+---------+---------+
|window                                    |store   |liczba_tx|suma_PLN |
+------------------------------------------+--------+---------+---------+
|{2026-04-12 08:00:00, 2026-04-12 08:30:00}|Gdańsk  |252      |93391.22 |
|{2026-04-12 08:00:00, 2026-04-12 08:30:00}|Kraków  |289      |117786.42|
|{2026-04-12 08:00:00, 2026-04-12 08:30:00}|Warszawa|275      |88441.58 |
|{2026-04-12 08:00:00, 2026-04-12 08:30:00}|Wrocław |296      |111540.59|
|{2026-04-12 08:30:00, 2026-04-12 09:00:00}|Gdańsk  |514      |209187.85|
|{2026-04-12 08:30:00, 2026-04-12 09:00:00}|Kraków  |532      |223541.41|
|{2026-04-12 08:30:00, 2026-04-12 09:00:00}|Warszawa|490      |182435.06|
|{2026-04-12 08:30:00, 2026-04-12 09:00:00}|Wrocław |502      |215587.17|
|{2026-04-12 09:00:00, 2026-04-12 09:30:00}|Gdańsk  |619      |253364.95|
|{2026-04-12 09:00:00, 2026-04-12 09:30:00}|Kraków  |590      |224358.03|
|{2026-04-12 09:00:00, 2026-04-12 09:3

In [13]:
from pyspark.sql.functions import window, sum as _sum, round as _round, desc

krakow_hourly_revenue = (
    df.filter(df.store == "Kraków")
    .groupBy(window("timestamp", "1 hour"))
    .agg(_round(_sum("amount"), 2).alias("przychod_PLN"))
    .orderBy(desc("przychod_PLN"))
)

krakow_hourly_revenue.show(1, truncate=False)

+------------------------------------------+------------+
|window                                    |przychod_PLN|
+------------------------------------------+------------+
|{2026-04-12 09:00:00, 2026-04-12 10:00:00}|483309.86   |
+------------------------------------------+------------+
only showing top 1 row



In [14]:
sliding = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))  # szerokość 1h, krok 30min
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .orderBy("od")
)
sliding.show(truncate=False)

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-04-12 07:30:00|2026-04-12 08:30:00|1112     |411159.81 |
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150     |1241911.3 |
|2026-04-12 08:30:00|2026-04-12 09:30:00|4443     |1753033.6 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661     |1896230.21|
|2026-04-12 09:30:00|2026-04-12 10:30:00|3696     |1557641.39|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189     |873403.24 |
|2026-04-12 10:30:00|2026-04-12 11:30:00|749      |289709.95 |
+-------------------+-------------------+---------+----------+



In [15]:
from pyspark.sql.functions import window, count

tumbling_rows = (
    df.groupBy(window("timestamp", "1 hour"))
    .agg(count("tx_id"))
    .count()
)
sliding_rows = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))
    .agg(count("tx_id"))
    .count()
)
print(f"Tumbling (1h):          {tumbling_rows} okien")
print(f"Sliding  (1h / 30min):  {sliding_rows} okien")

# Odpowiedz w komentarzu: dlaczego sliding ma więcej wierszy?
# Sliding ma więcej wierszy, bo okna nachodzą na siebie (krok 30 min < szerokość 1 h),
# więc jedna transakcja może trafić do 2 okien, a liczba unikalnych okien rośnie.

Tumbling (1h):          3 okien
Sliding  (1h / 30min):  7 okien


In [ ]:
# Część 5: Pytania kontrolne
# Odpowiedz na pytania w komentarzach:

# 1. Ile transakcji jest w oknie 09:00-10:00?
#    ODPOWIEDŹ: 4661

# 2. Jaka jest różnica między groupBy("store") a groupBy(window(...), "store")?
#    ODPOWIEDŹ: groupBy("store") agreguje po sklepie w całym zakresie czasu,
#    a groupBy(window(...), "store") agreguje po sklepie osobno w każdym oknie czasowym.

# 3. W oknie sliding 1h/30min - ile okien zawiera transakcje z godziny 09:30?
#    ODPOWIEDŹ: 2 okna

ODPOWIEDŹ 1 (09:00-10:00): 4661


In [17]:
from pyspark.sql.functions import window, avg, count, col, round as _round, desc, hour, minute

# 1) Godzina z najnizsza srednia kwota transakcji dla sklepu Gdansk
gdansk_lowest_avg_hour = (
    df.filter(col("store") == "Gdańsk")
    .groupBy(window("timestamp", "1 hour"))
    .agg(_round(avg("amount"), 2).alias("srednia_kwota_PLN"))
    .orderBy("srednia_kwota_PLN")
)

print("1) Gdansk - godzina z najnizsza srednia kwota:")
gdansk_lowest_avg_hour.show(1, truncate=False)

# 2) Liczba transakcji per kategoria w oknie 09:00-09:30
category_09_00_09_30 = (
    df.groupBy(window("timestamp", "30 minutes"), "category")
    .agg(count("tx_id").alias("liczba_tx"))
    .filter((hour(col("window.start")) == 9) & (minute(col("window.start")) == 0))
    .select("category", "liczba_tx")
    .orderBy("category")
)

print("2) Transakcje per kategoria w oknie 09:00-09:30:")
category_09_00_09_30.show(truncate=False)

# 3) Szczyt transakcji w oknach 15-minutowych (wszystkie sklepy)
peak_15m = (
    df.groupBy(window("timestamp", "15 minutes"))
    .agg(count("tx_id").alias("liczba_tx"))
    .orderBy(desc("liczba_tx"))
)

print("3) Szczyt transakcji - okno 15-minutowe:")
peak_15m.show(1, truncate=False)

1) Gdansk - godzina z najnizsza srednia kwota:
+------------------------------------------+-----------------+
|window                                    |srednia_kwota_PLN|
+------------------------------------------+-----------------+
|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|395.01           |
+------------------------------------------+-----------------+
only showing top 1 row

2) Transakcje per kategoria w oknie 09:00-09:30:
+-----------+---------+
|category   |liczba_tx|
+-----------+---------+
|elektronika|611      |
|książki    |622      |
|odzież     |605      |
|żywność    |567      |
+-----------+---------+

3) Szczyt transakcji - okno 15-minutowe:
+------------------------------------------+---------+
|window                                    |liczba_tx|
+------------------------------------------+---------+
|{2026-04-12 09:15:00, 2026-04-12 09:30:00}|1234     |
+------------------------------------------+---------+
only showing top 1 row

